# Hubmicroo Voice Assistant — run on Kaggle GPU

This notebook runs the **whole assistant on Kaggle's free GPU**, so it's fast (seconds per answer instead of minutes on a CPU laptop). It then gives you a **public HTTPS link** you can open in your browser and use the voice + text chat.

## Before you run (important)
1. Right sidebar → **Settings** → **Accelerator** = `GPU T4 x2` (or P100).
2. Right sidebar → **Settings** → **Internet** = **On**.
3. Then **Run All**. The last cell prints your public link.

> Note: Kaggle is for **testing/demoing on a GPU**. For the live client site, run the same project on the client's own server (see README / INTEGRATION.md).

In [ ]:
# 1) Install + start Ollama (the self-hosted LLM engine). Uses the GPU automatically.
import subprocess, time
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.Popen('ollama serve > /tmp/ollama.log 2>&1', shell=True)
time.sleep(8)
print(subprocess.run('curl -s http://127.0.0.1:11434/api/tags', shell=True,
                     capture_output=True, text=True).stdout or 'Ollama starting...')

In [ ]:
# 2) Download the models (Kaggle internet is fast). qwen2.5 speaks EN/UR/AR well.
#    Want sharper answers and the GPU has room? change 3b -> 7b here AND in cells 4 & 5.
!ollama pull qwen2.5:3b
!ollama pull bge-m3
!ollama list

In [ ]:
# 3) Get the project code + install Python deps.
%cd /kaggle/working
!rm -rf Hubmicroo-AI-Voice-Assistant
!git clone https://github.com/ai-with-abdullah/Hubmicroo-AI-Voice-Assistant.git
!pip install -q -r Hubmicroo-AI-Voice-Assistant/requirements.txt

In [ ]:
# 4) Build the search index (products + website pages). Fast on GPU.
import subprocess
out = subprocess.run(
    'cd /kaggle/working/Hubmicroo-AI-Voice-Assistant/backend && '
    'LLM_MODEL=qwen2.5:3b EMBED_MODEL=bge-m3 '
    'python -c "from app import indexer; print(indexer.reindex_all())"',
    shell=True, capture_output=True, text=True)
print(out.stdout); print(out.stderr[-500:] if out.stderr else '')

In [ ]:
# 5) Start the assistant server (FastAPI).
import subprocess, time
subprocess.Popen(
    'cd /kaggle/working/Hubmicroo-AI-Voice-Assistant/backend && '
    'LLM_MODEL=qwen2.5:3b EMBED_MODEL=bge-m3 OLLAMA_TIMEOUT=300 '
    'uvicorn app.main:app --host 0.0.0.0 --port 8000 > /tmp/server.log 2>&1',
    shell=True)
time.sleep(10)
print(subprocess.run('curl -s http://127.0.0.1:8000/api/health', shell=True,
                     capture_output=True, text=True).stdout)

In [ ]:
# 6) Open a PUBLIC HTTPS link to the assistant (free, no signup).
import subprocess, time, re
subprocess.run('wget -q -O /tmp/cloudflared '
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
    '&& chmod +x /tmp/cloudflared', shell=True)
subprocess.Popen('/tmp/cloudflared tunnel --url http://localhost:8000 > /tmp/cf.log 2>&1', shell=True)
time.sleep(14)
log = open('/tmp/cf.log').read()
m = re.search(r'https://[-\w]+\.trycloudflare\.com', log)
print('=' * 60)
print('OPEN THIS LINK IN YOUR BROWSER:', m.group(0) if m else 'NOT READY — re-run this cell')
print('=' * 60)
print('Tap the mic and speak (English / Urdu / Arabic), or type.')

In [ ]:
# 7) Optional quick test from inside the notebook.
!curl -s -X POST http://127.0.0.1:8000/api/chat -H 'Content-Type: application/json' \
  -d '{"message":"Do you have wireless headphones under 5000?"}'

## Put it on the real hubmicroo.com website
Use the public link from cell 6 in these 3 lines on every page of the site:
```html
<link rel="stylesheet" href="https://YOUR-LINK.trycloudflare.com/static/assistant.css">
<script>window.HUBMICROO_API_BASE = "https://YOUR-LINK.trycloudflare.com";</script>
<script src="https://YOUR-LINK.trycloudflare.com/static/assistant.js"></script>
```
(The trycloudflare link is temporary — only while this notebook runs. For production, deploy on the client's server, see `INTEGRATION.md`.)